### install the dependencies

In [3]:
!pip install transformers datasets sentencepiece accelerate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 87.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 753.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

### Check the device CPU or GPU

In [1]:
import tensorflow as tf

device_name = tf.test.gpu_device_name()
if device_name:
    print("✅ GPU in use:", device_name)
else:
    print("⚠️ Using CPU")


✅ GPU in use: /device:GPU:0


### Load the Spider dataset

In [2]:
from datasets import load_dataset

# Spider contains complex SQL with schema information
dataset = load_dataset("spider", trust_remote_code=True)
print(dataset)
print(dataset["train"][0])

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/5.51k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/831k [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/126k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1034 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['db_id', 'query', 'question', 'query_toks', 'query_toks_no_value', 'question_toks'],
        num_rows: 7000
    })
    validation: Dataset({
        features: ['db_id', 'query', 'question', 'query_toks', 'query_toks_no_value', 'question_toks'],
        num_rows: 1034
    })
})
{'db_id': 'department_management', 'query': 'SELECT count(*) FROM head WHERE age  >  56', 'question': 'How many heads of the departments are older than 56 ?', 'query_toks': ['SELECT', 'count', '(', '*', ')', 'FROM', 'head', 'WHERE', 'age', '>', '56'], 'query_toks_no_value': ['select', 'count', '(', '*', ')', 'from', 'head', 'where', 'age', '>', 'value'], 'question_toks': ['How', 'many', 'heads', 'of', 'the', 'departments', 'are', 'older', 'than', '56', '?']}


### Format Spider Data for T5

In [3]:
def format_spider_for_t5(example):
    input_text = f"translate English to SQL: {example['question']}"
    target_text = example["query"]
    return {"input_text": input_text, "target_text": target_text}

formatted_dataset = dataset.map(format_spider_for_t5)
formatted_dataset


Map:   0%|          | 0/7000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1034 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['db_id', 'query', 'question', 'query_toks', 'query_toks_no_value', 'question_toks', 'input_text', 'target_text'],
        num_rows: 7000
    })
    validation: Dataset({
        features: ['db_id', 'query', 'question', 'query_toks', 'query_toks_no_value', 'question_toks', 'input_text', 'target_text'],
        num_rows: 1034
    })
})

### Tokenize Inputs & Targets for T5

In [4]:
from transformers import T5Tokenizer

# Load T5 tokenizer
tokenizer = T5Tokenizer.from_pretrained("t5-small")

def tokenize_spider(example):
    model_inputs = tokenizer(
        example["input_text"],
        truncation=True,
        padding="max_length",
        max_length=256,
    )

    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            example["target_text"],
            truncation=True,
            padding="max_length",
            max_length=128,
        )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Apply tokenization
tokenized_spider = formatted_dataset.map(tokenize_spider, batched=True)


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


Map:   0%|          | 0/7000 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:3980: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/1034 [00:00<?, ? examples/s]

### Fine-Tune T5 on the Spider Dataset

In [5]:
from transformers import T5ForConditionalGeneration, TrainingArguments, Trainer, DataCollatorForSeq2Seq

# Load model
model = T5ForConditionalGeneration.from_pretrained("t5-small")

# Define training config
training_args = TrainingArguments(
    output_dir="./t5_spider_model",
    evaluation_strategy="epoch",
    learning_rate=3e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=2,
    weight_decay=0.01,
    save_strategy="epoch",
    logging_dir="./logs",
)

# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_spider["train"],
    eval_dataset=tokenized_spider["validation"],
    tokenizer=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model),
)

# Start fine-tuning
trainer.train()


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
<ipython-input-5-0bcad9c8bd03>:20: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


...

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: charishmagarikapati02 (charishmagarikapati02-northeastern-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss
1,0.252200,0.439894
2,0.177800,0.429925


TrainOutput(global_step=3500, training_loss=0.28924584089006694, metrics={'train_runtime': 549.512, 'train_samples_per_second': 25.477, 'train_steps_per_second': 6.369, 'total_flos': 947392610304000.0, 'train_loss': 0.28924584089006694, 'epoch': 2.0})

### Save the model

In [6]:
model.save_pretrained("t5_spider_finetuned")
tokenizer.save_pretrained("t5_spider_finetuned")


('t5_spider_finetuned/tokenizer_config.json',
 't5_spider_finetuned/special_tokens_map.json',
 't5_spider_finetuned/spiece.model',
 't5_spider_finetuned/added_tokens.json')

### Zip and Download the Folder

In [7]:
!zip -r t5_spider_finetuned.zip t5_spider_finetuned
from google.colab import files
files.download("t5_spider_finetuned.zip")

  adding: t5_spider_finetuned/ (stored 0%)
  adding: t5_spider_finetuned/generation_config.json (deflated 29%)
  adding: t5_spider_finetuned/special_tokens_map.json (deflated 85%)
  adding: t5_spider_finetuned/model.safetensors (deflated 8%)
  adding: t5_spider_finetuned/added_tokens.json (deflated 83%)
  adding: t5_spider_finetuned/spiece.model (deflated 48%)
  adding: t5_spider_finetuned/tokenizer_config.json (deflated 94%)
  adding: t5_spider_finetuned/config.json (deflated 63%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Test SQL Generation from Fine-Tuned Model

In [8]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

# Load your fine-tuned model
model_path = "/content/t5_spider_finetuned"
tokenizer = T5Tokenizer.from_pretrained(model_path)
model = T5ForConditionalGeneration.from_pretrained(model_path)

# Function to generate SQL from question
def predict_sql(question):
    input_text = f"translate English to SQL: {question}"
    input_ids = tokenizer(input_text, return_tensors="pt").input_ids

    output_ids = model.generate(input_ids, max_length=128, num_beams=4, early_stopping=True)
    sql_query = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return sql_query


In [11]:
question = "List the employees whose department is Data Scientist"
print("SQL:", predict_sql(question))


SQL: SELECT T1.employee_name FROM department AS T1 JOIN department AS T2 ON T1.department_id = T2.department_id WHERE T2.dept_name = 'Data Scientist'


### Upload CSV to Colab to test few samples through CSV

In [26]:
from google.colab import files
uploaded = files.upload()


Saving new_test_questions.csv to new_test_questions.csv


### Run Predictions on All Questions

In [25]:
import pandas as pd

# Load CSV
df = pd.read_csv("questions.csv")

# Predict function from earlier
def predict_sql(question):
    input_text = f"translate English to SQL: {question}"
    input_ids = tokenizer(input_text, return_tensors="pt").input_ids
    output_ids = model.generate(input_ids, max_length=128, num_beams=4, early_stopping=True)
    sql_query = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return sql_query

# Add predictions
df["generated_sql"] = df["question"].apply(predict_sql)

# Show result
df.head()


RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu! (when checking argument for argument index in method wrapper_CUDA__index_select)

### Download the Result as CSV

In [14]:
df.to_csv("sql_predictions.csv", index=False)
files.download("sql_predictions.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Re-train the model for better performance by adding few custom queries (Optional)

In [16]:
import pandas as pd

# Load your uploaded file
custom_df = pd.read_csv("custom_examples.csv")

# Convert headers + question into T5-style input_text
def make_input(row):
    return f"translate English to SQL: {row['question']} table: {row['headers']}"

custom_df["input_text"] = custom_df.apply(make_input, axis=1)
custom_df["target_text"] = custom_df["target_sql"]

custom_df[["input_text", "target_text"]].head()


,input_text,target_text
0,translate English to SQL: What is the highest ...,SELECT MAX(salary) FROM table WHERE department...
1,translate English to SQL: Give me the average ...,SELECT AVG(age) FROM table WHERE department = ...
2,translate English to SQL: How many employees a...,SELECT COUNT(id) FROM table WHERE department =...
3,translate English to SQL: List all products pr...,SELECT product_name FROM table WHERE price > 1...
4,translate English to SQL: What is the average ...,SELECT AVG(order_value) FROM table WHERE year ...


In [17]:
from datasets import Dataset

# Only keep columns T5 needs
custom_dataset = Dataset.from_pandas(custom_df[["input_text", "target_text"]])


In [19]:
from datasets import concatenate_datasets, DatasetDict

# Select Spider's input + target columns
spider_train = formatted_dataset["train"].select_columns(["input_text", "target_text"])

# Combine Spider and custom examples
combined_train = concatenate_datasets([spider_train, custom_dataset])

# Wrap it into a full dataset with validation
final_dataset = DatasetDict({
    "train": combined_train,
    "validation": formatted_dataset["validation"].select_columns(["input_text", "target_text"])
})


In [20]:
# Reuse your existing tokenizer
def tokenize_combined(example):
    model_inputs = tokenizer(
        example["input_text"],
        padding="max_length",
        truncation=True,
        max_length=256
    )
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            example["target_text"],
            padding="max_length",
            truncation=True,
            max_length=128
        )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Tokenize final_dataset
tokenized_final = final_dataset.map(tokenize_combined, batched=True)


Map:   0%|          | 0/7010 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:3980: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/1034 [00:00<?, ? examples/s]

In [21]:
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

# Update training args (you can change output_dir to save separately if desired)
training_args = TrainingArguments(
    output_dir="./t5_spider_enhanced",
    evaluation_strategy="epoch",
    learning_rate=3e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=5,  # You can increase if you want
    weight_decay=0.01,
    save_strategy="epoch",
    logging_dir="./logs",
)

# Create a new Trainer instance
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_final["train"],
    eval_dataset=tokenized_final["validation"],
    tokenizer=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model),
)

# Continue fine-tuning 🎯
trainer.train()


/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
<ipython-input-21-f82ea1c4b374>:17: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss
1,0.156300,0.467110
2,0.113700,0.576858
3,0.093200,0.572009
4,0.076200,0.594272
5,0.067000,0.583722


TrainOutput(global_step=8765, training_loss=0.10613053316942161, metrics={'train_runtime': 1165.1187, 'train_samples_per_second': 30.083, 'train_steps_per_second': 7.523, 'total_flos': 2371865070796800.0, 'train_loss': 0.10613053316942161, 'epoch': 5.0})

In [22]:
model.save_pretrained("t5_spider_enhanced_finetuned")
tokenizer.save_pretrained("t5_spider_enhanced_finetuned")


('t5_spider_enhanced_finetuned/tokenizer_config.json',
 't5_spider_enhanced_finetuned/special_tokens_map.json',
 't5_spider_enhanced_finetuned/spiece.model',
 't5_spider_enhanced_finetuned/added_tokens.json')

In [23]:
!zip -r t5_spider_enhanced_finetuned.zip t5_spider_enhanced_finetuned
from google.colab import files
files.download("t5_spider_enhanced_finetuned.zip")


  adding: t5_spider_enhanced_finetuned/ (stored 0%)
  adding: t5_spider_enhanced_finetuned/generation_config.json (deflated 29%)
  adding: t5_spider_enhanced_finetuned/special_tokens_map.json (deflated 85%)
  adding: t5_spider_enhanced_finetuned/model.safetensors (deflated 7%)
  adding: t5_spider_enhanced_finetuned/added_tokens.json (deflated 83%)
  adding: t5_spider_enhanced_finetuned/spiece.model (deflated 48%)
  adding: t5_spider_enhanced_finetuned/tokenizer_config.json (deflated 94%)
  adding: t5_spider_enhanced_finetuned/config.json (deflated 63%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [28]:
import pandas as pd
import torch

# Load CSV
df = pd.read_csv("/content/new_test_questions.csv")

# Setup device (GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [29]:
def predict_sql(question):
    input_text = f"translate English to SQL: {question}"
    input_ids = tokenizer(input_text, return_tensors="pt").input_ids.to(device)
    output_ids = model.generate(input_ids, max_length=128, num_beams=4, early_stopping=True)
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

# Generate predictions
df["generated_sql"] = df["question"].apply(predict_sql)


In [30]:
df.head(10)


,question,generated_sql
0,How many employees earn more than 80000?,SELECT count(*) FROM employees WHERE salary > ...
1,List the names of employees in the HR department,SELECT name FROM employees WHERE department_id...
2,What is the average salary for developers in 2...,SELECT avg(salary) FROM developers WHERE YEAR ...
3,Which customers placed orders in January?,SELECT T1.customer_name FROM customers AS T1 J...
4,Show all stadiums where attendance is below 40000,SELECT name FROM stadium WHERE attendance 40000
5,Get the revenue generated by online sales,"SELECT revenue FROM sales WHERE sales_type = ""..."
6,Count the number of employees who joined befor...,SELECT count(*) FROM employees WHERE join_date...
7,List all departments with more than 50 employees,SELECT dept_name FROM employees GROUP BY dept_...
8,What are the names of products costing between...,SELECT product_name FROM products WHERE price ...
9,Which states have customers with pending orders?,SELECT state FROM customers WHERE order_status...


In [31]:
df.to_csv("enhanced_sql_predictions.csv", index=False)

# Download
files.download("enhanced_sql_predictions.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>